<a href="https://colab.research.google.com/github/DaviMillo/My-projects/blob/main/TCC_SSparaProdu%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Desenvolvimento de um sensor virtual para estimativa da pressão de fundo em
poços produtores usando dados do 3W Dataset e comparação de modelos de
aprendizado de máquina

#Bloco 1 — Instalar bibliotecas e conectar ao Drive



In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#Bloco 2 — Configuração do experimento
#### Por padrão, o código tenta modelar todos os canais de pressão, temperatura e vazão listados, desde que existam nos arquivos. Para uma primeira execução mais rápida, recomendo começar apenas com ["P-PDG"].

In [2]:
from pathlib import Path

# Ajuste se o arquivo estiver em outra pasta do Drive.
ZIP_PATH = Path("/content/drive/MyDrive/TCC/TCC_Dados/Dados de Produção/data3W.zip")

# Pasta de trabalho local do Colab.
WORK_DIR = Path("/content/TCC_3W")
CACHE_DIR = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "resultados"

# Frequência temporal após agregação.
# Um passo de previsão equivale a um intervalo desta frequência.
RESAMPLE_RULE = "10s"

# "forecast": prevê o próximo passo e pode usar o valor atual do alvo.
# "virtual_sensor": estima o alvo sem usar o próprio sinal como entrada.
MODE = "virtual_sensor"

# Em forecast, 1 = próximo intervalo.
# Em virtual_sensor, use 0 para estimar o alvo no instante atual.
HORIZON_STEPS = 0

# Lags usados como atributos, medidos em passos após o reamostramento.
LAGS = [0, 1, 3, 6]

# None = tentar todos os canais de processo definidos abaixo.
# Para uma primeira execução mais rápida: TARGETS = ["P-PDG"]
TARGETS = None

# Limite de amostras por arquivo/instância para controlar memória e tempo.
MAX_SAMPLES_PER_INSTANCE = 2000

# Validação cruzada agrupada adicional somente para P-PDG.
RUN_GROUP_CV_FOR_PPDG = True
CV_FOLDS = 3

RANDOM_STATE = 42

# Canais de processo documentados no 3W.
# Canais inexistentes no ZIP serão ignorados.
PROCESS_TARGETS = [
    "P-ANULAR",
    "P-JUS-BS",
    "P-JUS-CKGL",
    "P-JUS-CKP",
    "P-MON-CKGL",
    "P-MON-CKP",
    "P-MON-SDV-P",
    "P-PDG",
    "PT-P",
    "P-TPT",
    "QBS",
    "QGL",
    "T-JUS-CKP",
    "T-MON-CKP",
    "T-PDG",
    "T-TPT",
]

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Não encontrei o ZIP em: {ZIP_PATH}\n"
        "Confira o caminho no Drive ou envie data3W.zip diretamente nesta conversa."
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ZIP encontrado:", ZIP_PATH)
print("Modo:", MODE)
print("Frequência de análise:", RESAMPLE_RULE)
print("Horizonte em passos:", HORIZON_STEPS)

ZIP encontrado: /content/drive/MyDrive/TCC/TCC_Dados/Dados de Produção/data3W.zip
Modo: virtual_sensor
Frequência de análise: 10s
Horizonte em passos: 0


#Bloco 3 — Ler o ZIP, limpar timestamps e criar cache local
#### O cache evita descompactar novamente o ZIP para cada variável-alvo. O código lê Parquet, CSV e Excel; arquivos não reconhecidos ou sem timestamp são contabilizados como ignorados.

In [3]:
import io
import re
import shutil
import zipfile
import hashlib
import warnings
import numpy as np
import pandas as pd

SUPPORTED_EXTENSIONS = {".parquet", ".csv", ".xlsx", ".xls"}

def read_zip_member(zf, member_name):
    """Lê um arquivo suportado de dentro do ZIP."""
    raw = zf.read(member_name)
    suffix = Path(member_name).suffix.lower()

    if suffix == ".parquet":
        return [("default", pd.read_parquet(io.BytesIO(raw), engine="pyarrow"))]

    if suffix == ".csv":
        return [("default", pd.read_csv(io.BytesIO(raw)))]

    if suffix in {".xlsx", ".xls"}:
        sheets = pd.read_excel(io.BytesIO(raw), sheet_name=None)
        return [(str(sheet_name), sheet_df) for sheet_name, sheet_df in sheets.items()]

    return []

def find_timestamp_column(df):
    """Localiza a coluna temporal, aceitando variações comuns de nome."""
    candidates = {
        "timestamp", "time", "datetime", "date_time",
        "datahora", "data_hora", "date"
    }

    for col in df.columns:
        normalized = str(col).strip().lower().replace(" ", "_")
        if normalized in candidates:
            return col

    if isinstance(df.index, pd.DatetimeIndex):
        return None

    return None

def standardize_instance(df, source_name):
    """
    Converte uma tabela para:
    - índice temporal UTC;
    - colunas numéricas de sensores;
    - sem class/state como preditores.
    """
    df = df.copy()
    time_col = find_timestamp_column(df)

    if time_col is not None:
        timestamps = pd.to_datetime(df[time_col], errors="coerce", utc=True)
        df = df.drop(columns=[time_col])
    elif isinstance(df.index, pd.DatetimeIndex):
        timestamps = pd.to_datetime(df.index, errors="coerce", utc=True)
    else:
        return None, f"sem coluna temporal reconhecida: {source_name}"

    valid_time = timestamps.notna()
    df = df.loc[valid_time].copy()
    timestamps = timestamps.loc[valid_time]

    if df.empty:
        return None, f"sem timestamps válidos: {source_name}"

    # Não usar rótulos de evento/estado como variáveis de entrada do regressor.
    excluded = {"class", "state"}

    numeric_data = {}
    for col in df.columns:
        if str(col).strip().lower() in excluded:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        if values.notna().any():
            numeric_data[str(col)] = values.to_numpy()

    if not numeric_data:
        return None, f"sem sinais numéricos: {source_name}"

    numeric_df = pd.DataFrame(numeric_data, index=pd.DatetimeIndex(timestamps))
    # Valor inválido observado na auditoria deste ZIP do 3W.
    # Substituí-lo por NaN impede que contamine o alvo e os lags.
    numeric_df = numeric_df.replace(-1.180116e42, np.nan)
    # Também transforma infinitos em ausências.
    numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
    numeric_df = numeric_df.sort_index()


    # Timestamps duplicados são agregados pela média.
    numeric_df = numeric_df.groupby(level=0).mean(numeric_only=True)

    # Reamostragem em frequência regular para dar significado ao horizonte.
    if RESAMPLE_RULE:
        numeric_df = numeric_df.resample(RESAMPLE_RULE).mean()

    numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
    numeric_df = numeric_df.loc[:, ~numeric_df.columns.duplicated()]

    if numeric_df.empty:
        return None, f"sem dados após limpeza: {source_name}"

    return numeric_df, None

def build_cache(zip_path, cache_dir):
    """Extrai e padroniza cada arquivo compatível para um cache Parquet."""
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    counts = {
        "supported_members": 0,
        "cached_instances": 0,
        "skipped": 0,
    }
    skip_reasons = []

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [
            name for name in zf.namelist()
            if not name.endswith("/")
            and Path(name).suffix.lower() in SUPPORTED_EXTENSIONS
        ]

        counts["supported_members"] = len(members)

        for file_i, member_name in enumerate(members):
            try:
                tables = read_zip_member(zf, member_name)
            except Exception as exc:
                counts["skipped"] += 1
                skip_reasons.append(f"{member_name}: erro de leitura ({exc})")
                continue

            for sheet_name, table in tables:
                source = f"{member_name}::{sheet_name}"
                clean_df, error = standardize_instance(table, source)

                if error:
                    counts["skipped"] += 1
                    skip_reasons.append(error)
                    continue

                digest = hashlib.sha1(source.encode("utf-8")).hexdigest()[:10]
                cache_file = cache_dir / f"inst_{file_i:05d}_{digest}.parquet"

                try:
                    clean_df.to_parquet(cache_file, engine="pyarrow", compression="snappy")
                except Exception as exc:
                    counts["skipped"] += 1
                    skip_reasons.append(f"{source}: erro ao salvar cache ({exc})")
                    continue

                manifest.append({
                    "cache_file": str(cache_file),
                    "group_id": source,
                    "source": source,
                    "rows": len(clean_df),
                    "columns": "|".join(map(str, clean_df.columns)),
                })
                counts["cached_instances"] += 1

    manifest_df = pd.DataFrame(manifest)
    manifest_path = cache_dir / "manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)

    return manifest_df, counts, skip_reasons

manifest, read_counts, skip_reasons = build_cache(ZIP_PATH, CACHE_DIR)

print("Resumo da leitura:")
print(read_counts)
print("Instâncias em cache:", len(manifest))

if skip_reasons:
    print("\nAlguns itens ignorados:")
    for reason in skip_reasons[:10]:
        print("-", reason)
    if len(skip_reasons) > 10:
        print(f"... e mais {len(skip_reasons) - 10} itens.")

/tmp/ipykernel_1691/2307228606.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(df[time_col], errors="coerce", utc=True)


Resumo da leitura:
{'supported_members': 1984, 'cached_instances': 1984, 'skipped': 0}
Instâncias em cache: 1984


#Bloco 4 — Preparar amostras de previsão
#### As entradas são sinais numéricos e seus lags. class e state são excluídos para evitar que os rótulos de evento “entreguem” a resposta ao modelo. No modo virtual_sensor, o próprio canal-alvo é removido das entradas.

In [4]:
def make_xy_for_instance(df, target, mode, horizon_steps, lags):
    """Cria X e y para uma instância, sem cruzar limites entre arquivos."""
    if target not in df.columns:
        return None, None

    feature_cols = list(df.columns)

    if mode == "virtual_sensor":
        feature_cols = [c for c in feature_cols if c != target]
    elif mode != "forecast":
        raise ValueError("MODE precisa ser 'forecast' ou 'virtual_sensor'.")

    if not feature_cols:
        return None, None

    blocks = []
    for lag in lags:
        block = df[feature_cols].shift(lag).copy()
        block.columns = [f"{col}__lag{lag}" for col in feature_cols]
        blocks.append(block)

    X = pd.concat(blocks, axis=1)
    y = df[target].shift(-horizon_steps).rename("target")

    valid_y = y.notna() & np.isfinite(y)
    X = X.loc[valid_y]
    y = y.loc[valid_y]

    if X.empty:
        return None, None

    return X, y

def sample_evenly(X, y, max_samples):
    """Limita amostras uniformemente no tempo, sem alterar o horizonte."""
    if max_samples is None or len(X) <= max_samples:
        return X, y

    positions = np.linspace(0, len(X) - 1, max_samples, dtype=int)
    return X.iloc[positions], y.iloc[positions]

def build_dataset_for_target(target):
    """Monta dataset de uma variável-alvo usando os grupos/arquivos como unidade."""
    X_parts, y_parts, group_parts = [], [], []

    for _, row in manifest.iterrows():
        cache_file = Path(row["cache_file"])

        try:
            df = pd.read_parquet(cache_file, engine="pyarrow")
        except Exception as exc:
            warnings.warn(f"Falha ao ler cache {cache_file.name}: {exc}")
            continue

        X_i, y_i = make_xy_for_instance(
            df=df,
            target=target,
            mode=MODE,
            horizon_steps=HORIZON_STEPS,
            lags=LAGS,
        )

        if X_i is None:
            continue

        X_i, y_i = sample_evenly(X_i, y_i, MAX_SAMPLES_PER_INSTANCE)

        X_parts.append(X_i)
        y_parts.append(y_i)
        group_parts.extend([row["group_id"]] * len(y_i))

    if not X_parts:
        return None, None, None

    X_all = pd.concat(X_parts, axis=0, sort=False)
    y_all = pd.concat(y_parts, axis=0)
    groups = np.asarray(group_parts)

    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    y_all = pd.to_numeric(y_all, errors="coerce")

    valid = y_all.notna() & np.isfinite(y_all.to_numpy())
    X_all = X_all.loc[valid].reset_index(drop=True)
    y_all = y_all.loc[valid].reset_index(drop=True)
    groups = groups[valid.to_numpy()]

    return X_all, y_all, groups

if TARGETS is None:
    targets_to_run = PROCESS_TARGETS.copy()
else:
    targets_to_run = list(TARGETS)

print("Alvos solicitados:", targets_to_run)

Alvos solicitados: ['P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT']


#Bloco 5 — Treinar três modelos, comparar com baseline e calcular métricas
#### A separação principal mantém cada arquivo/instância inteiro no treino ou no teste. Isso evita dividir aleatoriamente linhas temporalmente vizinhas. Se houver várias instâncias do mesmo poço e não existir identificador de poço no arquivo, essa divisão não garante separação por poço; os resultados devem ser descritos como validação por instância.

In [5]:
#Remove alvos nulos ou não finitos e mostra os extremos para inspeção; não os exclui automaticamente.

def audit_target_data(X, y, groups, target):
    X = X.reset_index(drop=True).copy()
    y = pd.to_numeric(pd.Series(y).reset_index(drop=True), errors="coerce")
    groups = np.asarray(groups)

    # Mantém ausências nos preditores como NaN para o imputador.
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Remove apenas linhas cujo alvo não é válido.
    y_values = y.to_numpy(dtype=float)
    valid_y = np.isfinite(y_values)
    removed = int((~valid_y).sum())

    if removed:
        print(
            f"  Auditoria {target}: removendo {removed:,} linhas "
            "com alvo nulo ou não finito.",
            flush=True,
        )

    X = X.loc[valid_y].reset_index(drop=True)
    y = y.loc[valid_y].reset_index(drop=True)
    groups = groups[valid_y]

    if len(y) == 0:
        print(f"  Auditoria {target}: nenhum alvo válido.", flush=True)
        return None, None, None

    print(f"\n  Auditoria do alvo {target}:", flush=True)
    print(f"    Mínimo: {y.min():.6g}", flush=True)
    print(f"    Máximo: {y.max():.6g}", flush=True)
    print(f"    Mediana: {y.median():.6g}", flush=True)
    print(
        "    Quantis:",
        y.quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1]).to_dict(),
        flush=True,
    )

    # Mostra as observações com maior magnitude e as instâncias de origem.
    top_idx = np.argsort(np.abs(y.to_numpy()))[-10:][::-1]
    extremos = pd.DataFrame({
        "linha": top_idx,
        "valor_alvo": y.iloc[top_idx].to_numpy(),
        "instancia": groups[top_idx],
    })

    print("    Dez maiores valores absolutos do alvo:", flush=True)
    print(extremos.to_string(index=False), flush=True)

    print("    Maiores magnitudes entre os preditores:", flush=True)
    print(
        X.abs().max(axis=0, skipna=True)
         .sort_values(ascending=False)
         .head(10)
         .to_string(),
        flush=True,
    )

    return X, y, groups

In [6]:
# ============================================================
# Seleção do alvo, treino, teste e comparação
# ============================================================

from time import perf_counter

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# ------------------------------------------------------------
# 1) OPÇÕES DE EXECUÇÃO
# ------------------------------------------------------------

# Opções para o TARGET_RUN_MODE:
# "one"  = roda somente TARGET_TO_RUN (Para testar rapidamente apenas a pressão de fundo)
# "best" = escolhe automaticamente o alvo com maior cobertura estimada
# "all"  = roda todos os alvos em targets_to_run (demora +5h)
TARGET_RUN_MODE = "best"

# Usado somente quando TARGET_RUN_MODE = "one"
TARGET_TO_RUN = "P-PDG"

# Mantém os parâmetros dos três modelos.
MODELS = {
    "RandomForest": RandomForestRegressor(
        n_estimators=50,
        max_features=0.8,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=60,
        max_features=0.8,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=80,
        learning_rate=0.1,
        max_leaf_nodes=20,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
}

# A escalação foi retirada do pipeline porque os três modelos são baseados
# em árvores. A imputação de valores ausentes continua sendo aplicada.
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", clone(model)),
    ])

def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Bias": float(np.mean(y_pred - y_true)),
        "R2": r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
    }

def unit_for_target(target):
    if target.startswith(("P-", "PT-")):
        return "Pa"
    if target.startswith("Q"):
        return "unidade da base"
    if target.startswith("T-"):
        return "unidade da base"
    return "unidade a confirmar"

def audit_target_data(X, y, groups, target):
    """
    Remove linhas cujo alvo é nulo ou não finito.
    Remove atributos completamente vazios.
    Mostra estatísticas do alvo para ajudar a identificar valores extremos.

    Não exclui automaticamente outliers: eles podem representar eventos
    reais e precisam ser verificados antes de serem removidos.
    """
    if X is None or y is None or groups is None:
        return None, None, None

    X = X.copy().reset_index(drop=True)
    y = pd.to_numeric(pd.Series(y).reset_index(drop=True), errors="coerce")
    groups = np.asarray(groups)

    # Garante que os atributos sejam numéricos e transforma infinitos em NaN.
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Remove colunas completamente vazias.
    nonempty_columns = X.columns[X.notna().any(axis=0)]
    X = X.loc[:, nonempty_columns]

    # Mantém somente linhas com alvo válido.
    y_values = y.to_numpy(dtype=float)
    valid_y = np.isfinite(y_values)
    removed = int((~valid_y).sum())

    if removed:
        print(
            f"  Auditoria {target}: removendo {removed:,} linhas "
            "com alvo nulo ou não finito.",
            flush=True,
        )

    X = X.loc[valid_y].reset_index(drop=True)
    y = y.loc[valid_y].reset_index(drop=True)
    groups = groups[valid_y]

    if len(y) == 0 or X.shape[1] == 0:
        print(
            f"  Auditoria {target}: não restaram dados ou atributos válidos.",
            flush=True,
        )
        return None, None, None

    quantiles = y.quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1])

    print(f"\n  Auditoria do alvo {target}:", flush=True)
    print(
        f"    Amostras válidas: {len(y):,} | "
        f"Atributos utilizáveis: {X.shape[1]:,} | "
        f"Instâncias: {len(np.unique(groups)):,}",
        flush=True,
    )
    print(
        "    Quantis do alvo:",
        quantiles.to_dict(),
        flush=True,
    )

    return X, y, groups

# ------------------------------------------------------------
# 2) RANQUEAR ALVOS PELO MANIFESTO
# ------------------------------------------------------------

def rank_targets_by_coverage(target_candidates):
    """
    Estima a cobertura de cada alvo usando o manifesto:
    - quantas instâncias têm a coluna;
    - quantas linhas essas instâncias contêm.

    É uma estimativa de disponibilidade, não de qualidade preditiva.
    """
    if "manifest" not in globals() or manifest is None or manifest.empty:
        print(
            "Aviso: manifesto indisponível; será usada a ordem de "
            "targets_to_run.",
            flush=True,
        )
        return list(target_candidates), pd.DataFrame()

    required_columns = {"columns", "rows"}
    if not required_columns.issubset(manifest.columns):
        print(
            "Aviso: o manifesto não tem as colunas 'columns' e 'rows'; "
            "será usada a ordem de targets_to_run.",
            flush=True,
        )
        return list(target_candidates), pd.DataFrame()

    manifest_local = manifest.copy()
    manifest_local["columns"] = (
        manifest_local["columns"].fillna("").astype(str)
    )
    manifest_local["rows"] = pd.to_numeric(
        manifest_local["rows"], errors="coerce"
    ).fillna(0)

    ranking = []

    for target in target_candidates:
        has_target = manifest_local["columns"].apply(
            lambda value: target in value.split("|")
        )

        rows_with_target = manifest_local.loc[has_target, "rows"]

        ranking.append({
            "Alvo": target,
            "Instancias_com_alvo": int(has_target.sum()),
            "Linhas_estimadas": int(rows_with_target.sum()),
        })

    ranking_df = pd.DataFrame(ranking)

    if ranking_df.empty:
        return list(target_candidates), ranking_df

    ranking_df = ranking_df.sort_values(
        ["Linhas_estimadas", "Instancias_com_alvo"],
        ascending=False,
    ).reset_index(drop=True)

    ordered_targets = ranking_df["Alvo"].tolist()
    return ordered_targets, ranking_df

# ------------------------------------------------------------
# 3) DEFINIR QUAIS ALVOS SERÃO EXECUTADOS
# ------------------------------------------------------------

if TARGET_RUN_MODE == "one":
    targets_for_run = [TARGET_TO_RUN]
    stop_after_first_valid_target = True

elif TARGET_RUN_MODE == "all":
    targets_for_run = list(targets_to_run)
    stop_after_first_valid_target = False

elif TARGET_RUN_MODE == "best":
    targets_for_run, target_ranking = rank_targets_by_coverage(
        list(targets_to_run)
    )
    stop_after_first_valid_target = True

    print(
        "\nAlvos mais disponíveis segundo o manifesto "
        "(estimativa de cobertura):",
        flush=True,
    )

    if not target_ranking.empty:
        print(
            target_ranking.head(10).to_string(index=False),
            flush=True,
        )

else:
    raise ValueError(
        "TARGET_RUN_MODE deve ser 'one', 'best' ou 'all'."
    )

print(
    f"\nModo selecionado: {TARGET_RUN_MODE}",
    flush=True,
)
print(
    f"Alvos na fila: {targets_for_run}",
    flush=True,
)

# ------------------------------------------------------------
# 4) TREINAMENTO E TESTE
# ------------------------------------------------------------

results = []
targets_available = []

total_targets = len(targets_for_run)

for target_number, target in enumerate(targets_for_run, start=1):
    target_start = perf_counter()

    print(
        f"\n[{target_number}/{total_targets}] "
        f"Iniciando preparação do alvo: {target}",
        flush=True,
    )

    prep_start = perf_counter()
    X, y, groups = build_dataset_for_target(target)

    if X is not None:
        X, y, groups = audit_target_data(X, y, groups, target)

    prep_elapsed = perf_counter() - prep_start

    if X is None or len(X) < 100:
        print(
            f"  Alvo ignorado: dados insuficientes ou canal ausente "
            f"({target}). Preparação: {prep_elapsed:.1f} s.",
            flush=True,
        )
        continue

    unique_groups = np.unique(groups)

    if len(unique_groups) < 2:
        print(
            f"  Alvo ignorado: precisa de pelo menos 2 instâncias "
            f"({target}); encontradas {len(unique_groups)}.",
            flush=True,
        )
        continue

    targets_available.append(target)

    print(
        f"  Preparação concluída em {prep_elapsed:.1f} s. "
        f"Amostras: {len(X):,} | "
        f"Instâncias: {len(unique_groups):,} | "
        f"Atributos: {X.shape[1]:,}",
        flush=True,
    )

    # Split por grupo: as amostras de cada grupo ficam juntas.
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(
        f"  Divisão concluída. "
        f"Treino: {len(train_idx):,} amostras; "
        f"teste: {len(test_idx):,} amostras.",
        flush=True,
    )

    # Baseline persistente somente em previsão temporal.
    if MODE == "forecast":
        persistence_col = f"{target}__lag0"

        if persistence_col in X_test.columns:
            baseline_pred = X_test[persistence_col].to_numpy(dtype=float)
            valid_baseline = np.isfinite(baseline_pred)

            if valid_baseline.any():
                baseline_metrics = calculate_metrics(
                    y_test.to_numpy()[valid_baseline],
                    baseline_pred[valid_baseline],
                )

                baseline_row = {
                    "Alvo": target,
                    "Modelo": "Baseline persistente",
                    "Unidade": unit_for_target(target),
                    "Amostras_teste": int(valid_baseline.sum()),
                    "Instancias_treino": int(
                        len(np.unique(groups[train_idx]))
                    ),
                    "Instancias_teste": int(
                        len(np.unique(groups[test_idx]))
                    ),
                    **baseline_metrics,
                    "CV_RMSE_média": np.nan,
                    "CV_RMSE_desvio": np.nan,
                }

                results.append(baseline_row)

                print(
                    f"  Baseline persistente calculado. "
                    f"RMSE: {baseline_row['RMSE']:.4g}",
                    flush=True,
                )

    # Treinar e avaliar cada modelo.
    for model_number, (model_name, model) in enumerate(
        MODELS.items(),
        start=1,
    ):
        model_start = perf_counter()

        print(
            f"\n  [{model_number}/{len(MODELS)}] "
            f"Iniciando treinamento: {target} / {model_name}",
            flush=True,
        )

        try:
            pipe = make_pipeline(model)

            fit_start = perf_counter()
            pipe.fit(X_train, y_train)
            fit_elapsed = perf_counter() - fit_start

            print(
                f"      Treinamento concluído em {fit_elapsed:.1f} s. "
                "Prevendo conjunto de teste...",
                flush=True,
            )

            predict_start = perf_counter()
            pred = pipe.predict(X_test)
            predict_elapsed = perf_counter() - predict_start

            metrics = calculate_metrics(y_test, pred)

            row = {
                "Alvo": target,
                "Modelo": model_name,
                "Unidade": unit_for_target(target),
                "Amostras_teste": len(y_test),
                "Instancias_treino": int(
                    len(np.unique(groups[train_idx]))
                ),
                "Instancias_teste": int(
                    len(np.unique(groups[test_idx]))
                ),
                **metrics,
                "CV_RMSE_média": np.nan,
                "CV_RMSE_desvio": np.nan,
            }

            print(
                f"      Previsão concluída em {predict_elapsed:.1f} s.",
                flush=True,
            )

            # Validação cruzada opcional, somente para P-PDG.
            if (
                RUN_GROUP_CV_FOR_PPDG
                and target == "P-PDG"
                and len(unique_groups) >= 2
            ):
                n_splits = min(CV_FOLDS, len(unique_groups))
                cv = GroupKFold(n_splits=n_splits)
                fold_rmses = []

                print(
                    f"      Iniciando validação cruzada: "
                    f"{n_splits} folds para {model_name}.",
                    flush=True,
                )

                for fold_number, (cv_train, cv_valid) in enumerate(
                    cv.split(X, y, groups=groups),
                    start=1,
                ):
                    fold_start = perf_counter()

                    print(
                        f"        Fold {fold_number}/{n_splits}: "
                        "treinando...",
                        flush=True,
                    )

                    cv_pipe = make_pipeline(model)
                    cv_pipe.fit(X.iloc[cv_train], y.iloc[cv_train])
                    cv_pred = cv_pipe.predict(X.iloc[cv_valid])

                    fold_rmse = np.sqrt(
                        mean_squared_error(y.iloc[cv_valid], cv_pred)
                    )
                    fold_rmses.append(fold_rmse)

                    print(
                        f"        Fold {fold_number}/{n_splits} concluído "
                        f"em {perf_counter() - fold_start:.1f} s. "
                        f"RMSE: {fold_rmse:.4g}",
                        flush=True,
                    )

                row["CV_RMSE_média"] = float(np.mean(fold_rmses))
                row["CV_RMSE_desvio"] = (
                    float(np.std(fold_rmses, ddof=1))
                    if len(fold_rmses) > 1
                    else 0.0
                )

            results.append(row)

            print(
                f"  Modelo concluído: {target} / {model_name} "
                f"| tempo: {perf_counter() - model_start:.1f} s "
                f"| MAE: {row['MAE']:.4g} "
                f"| RMSE: {row['RMSE']:.4g}",
                flush=True,
            )

        except Exception as exc:
            print(
                f"  ERRO no modelo {target} / {model_name}: "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )
            continue

    print(
        f"\n[{target_number}/{total_targets}] "
        f"Alvo concluído: {target} | "
        f"tempo total: {perf_counter() - target_start:.1f} s",
        flush=True,
    )

    # No modo "best", para após o primeiro alvo com dados suficientes.
    if stop_after_first_valid_target and target in targets_available:
        break

# ------------------------------------------------------------
# 5) EXIBIR RESULTADOS
# ------------------------------------------------------------

if results:
    results_df = pd.DataFrame(results)

    cols_to_show = [
        "Alvo",
        "Modelo",
        "Unidade",
        "Amostras_teste",
        "MAE",
        "MSE",
        "RMSE",
        "Bias",
        "R2",
        "CV_RMSE_média",
        "CV_RMSE_desvio",
    ]

    cols_to_show = [
        col for col in cols_to_show
        if col in results_df.columns
    ]

    print("\nTabela de resultados:")
    print(
        results_df[cols_to_show].to_markdown(
            index=False,
            floatfmt=".4g",
        )
    )
else:
    results_df = pd.DataFrame()
    print(
        "\nNenhum resultado foi gerado. "
        "Confira a disponibilidade dos alvos, a auditoria e os dados.",
        flush=True,
    )


Alvos mais disponíveis segundo o manifesto (estimativa de cobertura):
      Alvo  Instancias_com_alvo  Linhas_estimadas
     P-PDG                 1984           5093195
     P-TPT                 1984           5093195
 P-MON-CKP                 1913           4981245
 T-JUS-CKP                 1879           4923088
     T-TPT                 1769           4512516
       QGL                  746           1079168
P-JUS-CKGL                  650           1010685
  P-ANULAR                    0                 0
  P-JUS-BS                    0                 0
 P-JUS-CKP                    0                 0

Modo selecionado: best
Alvos na fila: ['P-PDG', 'P-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'T-TPT', 'QGL', 'P-JUS-CKGL', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP', 'T-PDG']

[1/16] Iniciando preparação do alvo: P-PDG

  Auditoria do alvo P-PDG:
    Amostras válidas: 3,267,744 | Atributos utilizáveis: 24 | Instâncias: 1,981
    Quanti

# Bloco 6 — Tabela Markdown e exportação


In [7]:
if not results:
    raise RuntimeError(
        "Nenhum resultado foi calculado. Verifique os arquivos do ZIP, "
        "os timestamps, as colunas e o resumo de leitura."
    )

results_df = pd.DataFrame(results)

# Ordena para facilitar a comparação dentro de cada alvo.
results_df = results_df.sort_values(
    ["Alvo", "RMSE"],
    ascending=[True, True],
).reset_index(drop=True)

# Salva métricas em CSV no Drive.
csv_path = OUTPUT_DIR / "metricas_comparacao_3w.csv"
results_df.to_csv(csv_path, index=False)

# Exibe tabela Markdown no notebook.
display_cols = [
    "Alvo",
    "Modelo",
    "Unidade",
    "Amostras_teste",
    "MAE",
    "RMSE",
    "Bias",
    "R2",
    "CV_RMSE_média",
    "CV_RMSE_desvio",
]

markdown_table = results_df[display_cols].to_markdown(
    index=False,
    floatfmt=".4g",
)

print(markdown_table)
print(f"\nCSV salvo em: {csv_path}")

| Alvo   | Modelo               | Unidade   |   Amostras_teste |       MAE |      RMSE |      Bias |     R2 |   CV_RMSE_média |   CV_RMSE_desvio |
|:-------|:---------------------|:----------|-----------------:|----------:|----------:|----------:|-------:|----------------:|-----------------:|
| P-PDG  | ExtraTrees           | Pa        |           661736 | 4.525e+05 | 2.778e+06 | 1.487e+05 | 0.9551 |       7.86e+07  |        4.78e+07  |
| P-PDG  | HistGradientBoosting | Pa        |           661736 | 1.039e+06 | 2.961e+06 | 5.874e+04 | 0.949  |       3.296e+07 |        3.105e+07 |
| P-PDG  | RandomForest         | Pa        |           661736 | 4.919e+05 | 3.185e+06 | 1.345e+05 | 0.941  |       7.98e+07  |        4.924e+07 |

CSV salvo em: /content/TCC_3W/resultados/metricas_comparacao_3w.csv


# Bloco 7 - Otimização

#### Aplicação de PCA/AG para maior eficiência dos dados analisados caso necessite